# Chapter 13: An integrated prediction project

Companion notebook for *Linear Algebra for Data Science and Machine Learning: Worked Examples and Python* by Dr. Sajjad Shaukat Jamal (Skills Outlet).

The cells below contain every Python example from Chapter 13, in the order they appear in the book. Work each example by hand first, then run the cell to check your result. Cells within a chapter build on each other, so run them from top to bottom.

## 13.5  Implement the calculation in Python

The companion notebook contains these functions. The core functions below make the fitted transformation explicit. k=None retains original centered features; k=1 uses the first PCA direction.

In [1]:
import numpy as np

def fit_model(X, y, k=None):
    mean = X.mean(axis=0)
    ymean = y.mean()
    Xc = X - mean
    if k is None:
        W = np.eye(X.shape[1])
    else:
        _, _, Vh = np.linalg.svd(Xc, full_matrices=False)
        W = Vh[:k].T
    Z = Xc @ W
    coef = np.linalg.lstsq(Z, y - ymean, rcond=None)[0]
    return mean, ymean, W, coef

def predict(model, X):
    mean, ymean, W, coef = model
    return ymean + (X - mean) @ W @ coef

def mse(y, pred):
    return np.mean((y - pred)**2)

## 13.7  Optional extension: Solving by gradient descent

### Example 10  Solve the full model by gradient descent

For the centered SSE, the Hessian is 2XcᵀXc, with eigenvalues 72 and 8. A constant step η = 0.02 lies between 0 and 2/72. Starting from zero slopes, iterate β ← β − 0.02[2Xcᵀ(Xcβ − yc)]. After 300 iterations the coefficients agree numerically with (1.5, 0.5). Recover intercept 25.

In [2]:
X = np.array([[13., 23.], [7., 17.], [11., 19.], [9., 21.]])
y = np.array([57., 45., 50., 48.])
Xc, yc = X - X.mean(axis=0), y - y.mean()
ridge = np.linalg.solve(Xc.T @ Xc + 4*np.eye(2),
                        Xc.T @ yc)
gd = np.zeros(2)
for step in range(300):
    gd -= .02 * (2 * Xc.T @ (Xc @ gd - yc))
direct = np.linalg.lstsq(Xc, yc, rcond=None)[0]
assert np.allclose(ridge, [1.15, .65])
assert np.allclose(gd, direct)